# Multi organ simulation motrpac hubmap 

⸻

## Section 1 — Load MoTrPAC Data

MoTrPAC typically provides:
	•	Pre vs Post exercise transcriptomics
	•	Proteomics
	•	Metabolomics
	•	Organ labels
	•	Timepoints


In [ ]:
import pandas as pd

motrpac = pd.read_csv("data/motrpac_processed.csv")

motrpac.head()

Columns might include:
	•	organ
	•	gene
	•	logFC
	•	pval
	•	timepoint

## Section 2 — Load HuBMAP Spatial Data

In [ ]:
from exerkinemap.datasets import load_spatial_data

hubmap_adata = load_spatial_data("hubmap_liver")

We extract:
	•	cell coordinates
	•	ligand expression
	•	receptor expression
	•	cell types


## Step 1 — Map MoTrPAC Signals to HuBMAP Cells

Key idea:

MoTrPAC gives organ-level deltas.
HuBMAP gives cell-level baseline.

We define:

x_i^{post} = x_i^{baseline} + \Delta_{organ(gene)}

Example implementation:

In [ ]:
def apply_motrpac_shift(adata, motrpac_df, organ_name):

    organ_data = motrpac_df[motrpac_df["organ"] == organ_name]

    gene_shift = dict(zip(organ_data["gene"], organ_data["logFC"]))

    for gene, delta in gene_shift.items():
        if gene in adata.var_names:
            adata[:, gene].X += delta

    return adata

Now HuBMAP becomes “exercise-aware”.

 ## Step 2 — Rebuild ExerkineMap per Organ

For each organ:
	1.	Construct spatial LRI graph
	2.	Compute diffusion
	3.	Extract features

Store per organ:

In [ ]:
organ_features = {}

for organ in ["muscle", "liver", "heart", "brain"]:
    adata_shifted = apply_motrpac_shift(hubmap_dict[organ], motrpac, organ)
    features = run_exerkinemap_pipeline(adata_shifted)
    organ_features[organ] = features